# Generador del Visor 3D d'Ocupació d'Aules (Correcció Excés Aforament)
Aquesta versió implementa la correcció lògica: l'ocupació es calcula cercant la setmana de màxima assistència, evitant falsos positius per sumar alumnes de setmanes diferents.

In [7]:
import pandas as pd
import json
import numpy as np

def convert_types(obj):
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    return str(obj)

print("1. Generant layout.json amb la nova distribució...")
aules = pd.read_csv('aules.csv', sep=None, engine='python')

col_aula_id = next((c for c in aules.columns if 'aula' in c.lower() or 'codi' in c.lower()), aules.columns[0])
col_cap = next((c for c in aules.columns if 'capacitat' in c.lower() or 'aforament' in c.lower()), aules.columns[-1])
col_desc = next((c for c in aules.columns if 'descripcio' in c.lower() or 'nom' in c.lower()), None)

building_z = {'Q4': -60, 'Q3': -20, 'Q6': -20, 'Q2': 20, 'Q5': 20, 'Q1': 60, 'QC': 100, 'J': 140, 'B3': 140}
building_x_base = {'Q1': -120, 'Q2': -120, 'Q3': -120, 'Q4': -120, 'Q5': 20, 'Q6': 20, 'QC': -100, 'J': -120, 'B3': 20}

rows_parsed = []
for _, row in aules.iterrows():
    aula_id = str(row[col_aula_id]).strip()
    if aula_id == 'nan' or not aula_id:
        continue
        
    cap = float(row[col_cap]) if pd.notna(row[col_cap]) else 999.0
    parts = aula_id.split('/')
    if len(parts) >= 2:
        edifici = parts[0]
        codi = parts[1]
        
        piso = 0
        if len(codi) >= 4 and codi[0].isdigit():
            piso = int(codi[0])
        elif len(codi) >= 3 and codi[0].isdigit():
            piso = int(codi[0])
        elif '.' in codi:
            piso = int(codi[0])
            
        num = int(''.join(filter(str.isdigit, codi))) if any(str.isdigit(ch) for ch in codi) else 0
        desc = str(row[col_desc]) if col_desc and pd.notna(row[col_desc]) else ''
        
        rows_parsed.append({
            'id': aula_id,
            'edifici': edifici,
            'codi': codi,
            'piso': piso,
            'num': num,
            'desc': desc,
            'capacitat': cap
        })

df_parsed = pd.DataFrame(rows_parsed).sort_values(by=['edifici', 'piso', 'num'])

layout_3d = []
for (edifici, piso), group in df_parsed.groupby(['edifici', 'piso']):
    z = int(building_z.get(edifici, -100))
    y = int(piso * 18)
    x_base = int(building_x_base.get(edifici, -100))
    
    for idx, (_, room) in enumerate(group.iterrows()):
        layout_3d.append({
            'id': str(room['id']),
            'x': int(x_base + (idx * 18)),
            'y': int(y),
            'z': int(z),
            'desc': str(room['desc']),
            'capacitat': float(room['capacitat'])
        })

with open('layout.json', 'w', encoding='utf-8') as f:
    json.dump(layout_3d, f, ensure_ascii=False, default=convert_types)

print("Finalitzat correctament sense errors de serialització!")

1. Generant layout.json amb la nova distribució...
Finalitzat correctament sense errors de serialització!


In [8]:
print("2. Processant sessions.csv (1r Semestre, Ocupació Màxima per Setmana)...")
sessions = pd.read_csv('sessions.csv', sep=None, engine='python')
s2 = sessions.copy()

if 'semestre' in s2.columns:
    s2_filtrat = s2[s2['semestre'].astype(str).str.contains('1')]
    if len(s2_filtrat) > 0:
        s2 = s2_filtrat

col_sess_aula = next((c for c in ['aula', 'codi_aula', 'aula_partida', 'espai'] if c in s2.columns), s2.columns[0])
s2 = s2.dropna(subset=[col_sess_aula])

def obtenir_blocs(row):
    try:
        h_str = str(row.get('h_ini', '0')).strip()
        if h_str == 'nan': return []
        
        if ':' in h_str:
            h = int(h_str.split(':')[0])
            m = int(h_str.split(':')[1])
        elif '.' in h_str:
            h = int(float(h_str))
            m = int((float(h_str) - h) * 60)
        else:
            h = int(h_str)
            m = 0
            
        durada = int(float(row.get('durada_min', 60)))
        
        start_mins = h * 60 + m
        end_mins = start_mins + durada
        
        start_h = start_mins // 60
        end_h = (end_mins - 1) // 60 
        
        blocs = []
        for current_h in range(start_h, end_h + 1):
            if 0 <= current_h < 24:
                blocs.append(f"{current_h:02d}:00")
        return blocs
    except Exception as e:
        return []

s2['blocs'] = s2.apply(obtenir_blocs, axis=1)
s2_exp = s2.explode('blocs')

def parse_dia(d):
    d = str(d).strip().upper()
    if 'L' in d or d == '1' or 'DILL' in d: return 'L'
    if ('M' in d and 'DI' not in d) or d == '2' or 'DIMAR' in d: return 'M'
    if 'X' in d or 'DIMEC' in d or d == '3' or d == 'DC': return 'X'
    if 'J' in d or 'DIJ' in d or d == '4' or d == 'DJ': return 'J'
    if 'V' in d or 'DIV' in d or d == '5' or d == 'DV': return 'V'
    return d

s2_exp['dia_net'] = s2_exp['dia'].apply(parse_dia) if 'dia' in s2_exp.columns else 'L'
s2_exp = s2_exp.dropna(subset=['blocs'])

col_alumnes = next((c for c in s2_exp.columns if any(x in c.lower() for x in ['alumn', 'matric', 'estud'])), None)
col_setmana = next((c for c in s2_exp.columns if 'setmana' in c.lower() or 'week' in c.lower()), None)

ocupacio = {'L':{}, 'M':{}, 'X':{}, 'J':{}, 'V':{}}
sessions_processades = 0

# NOU CÀLCUL: Primer agrupar per setmana i sumar
if col_setmana:
    # Si tenim columna de setmana, calculem l'ocupació concurrent
    agrupat_setmana = s2_exp.groupby(['dia_net', 'blocs', col_sess_aula, col_setmana])[col_alumnes].sum().reset_index() if col_alumnes else s2_exp.groupby(['dia_net', 'blocs', col_sess_aula, col_setmana]).size().reset_index(name='count')
    # Ara agafem el MÀXIM d'alumnes que hi ha hagut en qualsevol setmana
    agrupat_final = agrupat_setmana.groupby(['dia_net', 'blocs', col_sess_aula])[col_alumnes if col_alumnes else 'count'].max().reset_index()
else:
    # Fallback si no hi ha dades de setmana, utilitzar el mitjà per no excedir-se
    agrupat_final = s2_exp.groupby(['dia_net', 'blocs', col_sess_aula])[col_alumnes].mean().reset_index() if col_alumnes else s2_exp.groupby(['dia_net', 'blocs', col_sess_aula]).size().reset_index(name='count')

for _, row in agrupat_final.iterrows():
    dia = row['dia_net']
    hora = row['blocs']
    aula = str(row[col_sess_aula]).strip()
    alumnes_totals = row[col_alumnes if col_alumnes else 'count']
    
    if dia not in ocupacio:
        ocupacio[dia] = {}
    if hora not in ocupacio[dia]:
        ocupacio[dia][hora] = {}
        
    cap = capacitat_aules.get(aula, 999)
    
    if alumnes_totals > cap:
        ocupacio[dia][hora][aula] = 2
    else:
        ocupacio[dia][hora][aula] = 1
    sessions_processades += 1

with open('ocupacio.json', 'w', encoding='utf-8') as f:
    json.dump(ocupacio, f, ensure_ascii=False)

print(f"-> S'han guardat {sessions_processades} franges amb ocupació corregida a ocupacio.json")


2. Processant sessions.csv (1r Semestre, Ocupació Màxima per Setmana)...
-> S'han guardat 1539 franges amb ocupació corregida a ocupacio.json


C:\Users\arnau\AppData\Local\Temp\ipykernel_19348\1904454529.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  s2['blocs'] = s2.apply(obtenir_blocs, axis=1)


In [9]:
print("3. Generant el visor3d_corregit.html...")
html_template = """<!DOCTYPE html>\n<html lang=\"ca\">\n<head>\n    <meta charset=\"UTF-8\">\n    <title>Visor 3D Aules UAB</title>\n    <link href=\"https://fonts.googleapis.com/css2?family=Nunito:wght@400;700&display=swap\" rel=\"stylesheet\">\n    <style>\n        body { \n            margin: 0; \n            overflow: hidden; \n            font-family: 'Nunito', 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; \n            background-color: #636e72;\n        }\n        #ui { position: absolute; bottom: 30px; left: 50%; transform: translateX(-50%); background: rgba(255,255,255,0.95); padding: 15px 35px; border-radius: 12px; display: flex; gap: 25px; box-shadow: 0 8px 32px rgba(0,0,0,0.2); }\n        .control { display: flex; flex-direction: column; gap: 8px; font-weight: 700; color: #2d3436; }\n        select, input[type=range] { cursor: pointer; }\n        select { padding: 5px; border-radius: 5px; border: 1px solid #b2bec3; font-family: 'Nunito', sans-serif; font-weight: 700; }\n        #legend { position: absolute; top: 20px; right: 20px; background: rgba(255,255,255,0.95); padding: 15px 20px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.15); color: #2d3436; font-weight: 700; }\n        .leg-item { display: flex; align-items: center; gap: 12px; margin-bottom: 10px; font-size: 15px; }\n        .leg-item:last-child { margin-bottom: 0; }\n        .box { width: 18px; height: 18px; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }\n        #info { position: absolute; top: 20px; left: 20px; background: rgba(45, 52, 54, 0.9); color: white; padding: 15px 20px; border-radius: 12px; pointer-events: none; display: none; box-shadow: 0 4px 15px rgba(0,0,0,0.2); font-size: 15px; line-height: 1.5; }\n        #info strong { color: #74b9ff; font-size: 18px; }\n    </style>\n    <script src=\"https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js\"></script>\n    <script src=\"https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js\"></script>\n</head>\n<body>\n    <div id=\"legend\">\n        <div class=\"leg-item\"><div class=\"box\" style=\"background: #dfe6e9;\"></div>Lliure</div>\n        <div class=\"leg-item\"><div class=\"box\" style=\"background: #00b894;\"></div>Ocupada</div>\n        <div class=\"leg-item\"><div class=\"box\" style=\"background: #d63031;\"></div>Excés d'Aforament</div>\n    </div>\n    <div id=\"info\"></div>\n    <div id=\"ui\">\n        <div class=\"control\">\n            <label>Dia:</label>\n            <select id=\"dia\">\n                <option value=\"L\">Dilluns</option><option value=\"M\">Dimarts</option>\n                <option value=\"X\">Dimecres</option><option value=\"J\">Dijous</option>\n                <option value=\"V\">Divendres</option>\n            </select>\n        </div>\n        <div class=\"control\">\n            <label>Hora: <span id=\"hora-txt\">08:00</span></label>\n            <input type=\"range\" id=\"hora\" min=\"8\" max=\"20\" step=\"1\" value=\"8\" style=\"width: 200px;\">\n        </div>\n    </div>\n    <script>\n        let layout = [];\n        let ocupacio = {};\n        const rooms = {};\n\n        const scene = new THREE.Scene();\n        const camera = new THREE.PerspectiveCamera(50, window.innerWidth/window.innerHeight, 0.1, 1000);\n        camera.position.set(0, 150, 180);\n\n        const renderer = new THREE.WebGLRenderer({antialias: true, alpha: true});\n        renderer.setSize(window.innerWidth, window.innerHeight);\n        renderer.shadowMap.enabled = true;\n        renderer.shadowMap.type = THREE.PCFSoftShadowMap;\n        document.body.appendChild(renderer.domElement);\n        \n        const controls = new THREE.OrbitControls(camera, renderer.domElement);\n        controls.enableDamping = true;\n\n        const ambientLight = new THREE.AmbientLight(0xffffff, 0.85);\n        scene.add(ambientLight);\n        \n        const dirLight = new THREE.DirectionalLight(0xffffff, 0.4);\n        dirLight.position.set(50, 120, 80);\n        dirLight.castShadow = true;\n        scene.add(dirLight);\n\n        const geo = new THREE.BoxGeometry(14, 10, 14);\n        const terra = new THREE.Mesh(\n            new THREE.PlaneGeometry(350, 300), \n            new THREE.MeshLambertMaterial({color: 0x95a5a6})\n        );\n        terra.rotation.x = -Math.PI / 2;\n        terra.position.y = -5;\n        terra.receiveShadow = true;\n        scene.add(terra);\n\n        function createTextSprite(message) {\n            const canvas = document.createElement('canvas');\n            const context = canvas.getContext('2d');\n            canvas.width = 256; canvas.height = 128;\n            context.font = \"bold 32px 'Nunito', 'Segoe UI', Tahoma, sans-serif\";\n            context.shadowColor = \"rgba(255, 255, 255, 0.9)\";\n            context.shadowBlur = 6;\n            context.fillStyle = \"rgba(45, 52, 54, 1.0)\";\n            context.textAlign = \"center\";\n            context.textBaseline = \"middle\";\n            context.fillText(message, 128, 64);\n            const texture = new THREE.CanvasTexture(canvas);\n            const spriteMaterial = new THREE.SpriteMaterial({ map: texture, transparent: true });\n            const sprite = new THREE.Sprite(spriteMaterial);\n            sprite.scale.set(16, 8, 1);\n            return sprite;\n        }\n\n        function inicialitzarLayout(dataLayout) {\n            layout = dataLayout;\n            layout.forEach(aula => {\n                const mat = new THREE.MeshLambertMaterial({color: 0xdfe6e9});\n                const mesh = new THREE.Mesh(geo, mat);\n                mesh.position.set(aula.x, aula.y, aula.z);\n                mesh.castShadow = true;\n                mesh.receiveShadow = true;\n                mesh.userData = aula;\n                scene.add(mesh);\n                rooms[aula.id] = mesh;\n                \n                const edges = new THREE.EdgesGeometry(geo);\n                const line = new THREE.LineSegments(edges, new THREE.LineBasicMaterial({color: 0xb2bec3, linewidth: 2}));\n                mesh.add(line);\n                \n                const textSprite = createTextSprite(aula.id);\n                textSprite.position.y = 9;\n                mesh.add(textSprite);\n            });\n        }\n\n        function actualitzar() {\n            const dia = document.getElementById('dia').value;\n            const horaVal = document.getElementById('hora').value;\n            const horaStr = horaVal.padStart(2, '0') + ':00';\n            document.getElementById('hora-txt').textContent = horaStr;\n            \n            const dadesHora = (ocupacio[dia] && ocupacio[dia][horaStr]) ? ocupacio[dia][horaStr] : {};\n            \n            Object.keys(rooms).forEach(id => {\n                const estat = dadesHora[id] || 0;\n                let color = 0xdfe6e9;\n                if (estat === 1) color = 0x00b894;\n                if (estat === 2) color = 0xd63031;\n                rooms[id].material.color.setHex(color);\n            });\n        }\n\n        document.getElementById('dia').addEventListener('change', actualitzar);\n        document.getElementById('hora').addEventListener('input', actualitzar);\n\n        Promise.all([\n            fetch('layout.json').then(r => r.json()),\n            fetch('ocupacio.json').then(r => r.json())\n        ]).then(([layoutData, ocupacioData]) => {\n            ocupacio = ocupacioData;\n            inicialitzarLayout(layoutData);\n            actualitzar();\n        }).catch(err => console.error(\"Error carregant dades JSON. Comprova el servidor local.\", err));\n\n        const raycaster = new THREE.Raycaster();\n        const mouse = new THREE.Vector2();\n        const infoBox = document.getElementById('info');\n\n        window.addEventListener('mousemove', (e) => {\n            mouse.x = (e.clientX / window.innerWidth) * 2 - 1;\n            mouse.y = -(e.clientY / window.innerHeight) * 2 + 1;\n            raycaster.setFromCamera(mouse, camera);\n            const intersects = raycaster.intersectObjects(Object.values(rooms));\n            \n            if (intersects.length > 0) {\n                const data = intersects[0].object.userData;\n                infoBox.style.display = 'block';\n                infoBox.innerHTML = `<strong>${data.id}</strong><br>${data.desc}<br>Capacitat: ${data.capacitat}`;\n            } else {\n                infoBox.style.display = 'none';\n            }\n        });\n\n        function anim() {\n            requestAnimationFrame(anim);\n            controls.update();\n            renderer.render(scene, camera);\n        }\n        anim();\n    </script>\n</body>\n</html>\n"""
with open('visor3d_corregit.html', 'w', encoding='utf-8') as f:
    f.write(html_template)
print("Fet! Obrir 'visor3d_corregit.html' amb un servidor local per veure el resultat.")


3. Generant el visor3d_corregit.html...
Fet! Obrir 'visor3d_corregit.html' amb un servidor local per veure el resultat.
